In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
display(spark.table("ehr_pipeline.bronze.synthea_patients").limit(20))


In [0]:
patients = spark.table("ehr_pipeline.bronze.synthea_patients")

In [0]:
patients.groupBy('MARITAL').count().show()

In [0]:
patients.filter(F.col('id').isNotNull()).count()

1. Lowercase column names
2. Cast BIRTHDATE to date
3. Cast DEATHDATE to date
4. Derive patient_age from BIRTHDATE
5. Derive is_deceased flag from DEATHDATE
6. Decode MARITAL to readable label
7. Drop SSN, DRIVERS, PASSPORT
8. Deduplicate by Id
9. Keep _ingested_at and _source_file
10. Add _silver_processed_at audit column

In [0]:
# 1 Lowercase Column names

for column in patients.columns:
    patients = patients.withColumnRenamed(column, column.lower())

In [0]:
patients = patients.withColumn("birthdate", F.col("birthdate").cast("date"))
patients = patients.withColumn("deathdate", F.col("deathdate").cast("date"))

In [0]:
# 4. Derive patient_age from BIRTHDATE
patients = patients.withColumn("patient_age", F.datediff(F.current_date(), F.col("birthdate")) / 365)


In [0]:
# 5. Derive is_deceased flag from DEATHDATE
patients = patients.withColumn("is_deceased", F.when(F.col("deathdate").isNotNull(), True).otherwise(False))

In [0]:
patients = patients.withColumn("maritial", F.when(F.col("marital") == 'M', 'Married') \
            .when(F.col("marital") == 'S', 'Single') \
            .otherwise('unknown'))

In [0]:
# 7. Drop SSN, DRIVERS, PASSPORT

patient = patients.drop('ssn', 'drivers', 'passport')

In [0]:
# 8 Deduplicate by Id
patient = patient.dropDuplicates(['id'])


In [0]:
# Add _silver_processed_at audit column

patient = patient.withColumn("_silver_processed_at", F.current_timestamp())
# 9. Write to silver table
patient.write.mode("overwrite").saveAsTable("ehr_pipeline.silver.patient")

In [0]:
def transform_patients(spark, df):
    # lowercase column names
    df = df.toDF(*[c.lower() for c in df.columns])
    
    # handle dash values before casting
    df = df.withColumn("deathdate",
        F.when(F.col("deathdate") == "-", None)
         .otherwise(F.col("deathdate"))
    )
    
    # cast date columns
    df = df.withColumn("birthdate", F.col("birthdate").cast("date"))
    df = df.withColumn("deathdate", F.col("deathdate").cast("date"))
    
    # derive age and deceased flag
    df = df.withColumn(
        "patient_age",
        (F.datediff(F.current_date(), F.col("birthdate")) / 365).cast("integer")
    )
    df = df.withColumn(
        "is_deceased",
        F.when(F.col("deathdate").isNotNull(), True).otherwise(False)
    )
    
    # decode marital status
    df = df.withColumn(
        "marital_status",
        F.when(F.col("marital") == "M", "Married")
         .when(F.col("marital") == "S", "Single")
         .otherwise("Unknown")
    ).drop("marital")
    
    # drop PII columns
    df = df.drop("ssn", "drivers", "passport")
    
    # deduplicate — keep latest ingestion per patient
    window = Window.partitionBy("id").orderBy(F.col("_ingested_at").desc())
    df = df.withColumn("_row_rank", F.row_number().over(window)) \
           .filter(F.col("_row_rank") == 1) \
           .drop("_row_rank")
    
    # add silver audit column
    df = df.withColumn("_silver_processed_at", F.current_timestamp())
    
    return df